# Data Monitoring

In [31]:
import json
import sys
import time

import boto3
import sagemaker
from botocore.exceptions import ClientError
from sagemaker.model_monitor import (
    DefaultModelMonitor,
    CronExpressionGenerator,
)

sys.path.append('../config')
import config

In [32]:
# Set up AWS session
session = boto3.session.Session()
region = session.region_name
sagemaker_session = sagemaker.Session()
sm_client = boto3.client('sagemaker', region_name=region)
role = sagemaker.get_execution_role()
bucket = config.S3_BUCKET

In [14]:
# Set up naming vars
S3_MONITORING_PREFIX = 'final_project/monitoring/data-quality'
S3_REPORTS_PREFIX = f'{S3_MONITORING_PREFIX}/reports'
S3_BASELINE_PREFIX = f'{S3_MONITORING_PREFIX}/baseline'
S3_TRAINING_DATA_PREFIX = 'final_project/feature_engineer'
ENDPOINT_NAME = "final-model-endpoint"
TRAIN_DATA_PATH = f's3://{bucket}/{S3_TRAINING_DATA_PREFIX}/train.csv'
BASELINE_OUTPUT_URI = f's3://{bucket}/{S3_BASELINE_PREFIX}'
REPORTS_OUTPUT_URI = f's3://{bucket}/{S3_REPORTS_PREFIX}'
MONITOR_SCHEDULE_NAME = f'ddos-data-quality-schedule-{int(time.time())}'

In [5]:
# Create data quality monitor
data_quality_monitor = DefaultModelMonitor(
    role=role,
    instance_count=1,
    instance_type='ml.m5.large',
    volume_size_in_gb=10,
    max_runtime_in_seconds=3600,
    sagemaker_session=sagemaker_session,
)

In [6]:
# Baseline off of the training data
baselining_job = data_quality_monitor.suggest_baseline(
    job_name=f'data-quality-baseline-job-{int(time.time())}',
    baseline_dataset=TRAIN_DATA_PATH,
    dataset_format={'csv': {'header': False}},
    output_s3_uri=BASELINE_OUTPUT_URI,
    wait=True,
    logs=True,
)

INFO:sagemaker:Creating processing-job with name data-quality-baseline-job-1750290442


..............2025-06-18 23:49:55.610894: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2025-06-18 23:49:55.610927: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
2025-06-18 23:49:57.294121: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory
2025-06-18 23:49:57.294155: W tensorflow/stream_executor/cuda/cuda_driver.cc:269] failed call to cuInit: UNKNOWN ERROR (303)
2025-06-18 23:49:57.294217: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not appear to be running on this host (ip-10-0-161-211.ec2.internal): /proc/driver/nvidia/version does not exist
2025-06-18 23:49:57.294538: I tens

In [22]:
# Verify the endpoint is available
try:
    sm_client.describe_endpoint(EndpointName=ENDPOINT_NAME)
    print(f"Endpoint '{ENDPOINT_NAME}' is available.")
except ClientError as e:
    print(f"ERROR: Endpoint '{ENDPOINT_NAME}' is not available.")
    print(e)

Endpoint 'final-model-endpoint' is available.


In [23]:
# Run the data monitor every hour, saving the results to S3
data_quality_monitor.create_monitoring_schedule(
    monitor_schedule_name=MONITOR_SCHEDULE_NAME,
    endpoint_input=ENDPOINT_NAME,
    output_s3_uri=REPORTS_OUTPUT_URI,
    statistics=data_quality_monitor.latest_baselining_job.baseline_statistics(),
    constraints=data_quality_monitor.latest_baselining_job.suggested_constraints(),
    schedule_cron_expression=CronExpressionGenerator.hourly(),
    enable_cloudwatch_metrics=True,
)

INFO:sagemaker.model_monitor.model_monitoring:Creating Monitoring Schedule with name: ddos-data-quality-schedule-1750291218


# Infrastructure Monitoring

In [ ]:
# Set up AWS session
cloudwatch_client = session.client('cloudwatch')

In [21]:
# Set up naming vars
DASHBOARD_NAME = f"SageMaker-Endpoint-Health-{ENDPOINT_NAME}"

In [ ]:
# Set up dimensions for Cloudwatch alarms
sm_client.describe_endpoint(EndpointName=ENDPOINT_NAME)
endpoint_dimensions = [
    {'Name': 'EndpointName', 'Value': ENDPOINT_NAME},
    {'Name': 'VariantName', 'Value': 'AllTraffic'}
]

In [24]:
# Helper function for creating a new cloudwatch alarm
def create_cloudwatch_alarm(alarm_name, metric_name, namespace, dimensions, statistic, threshold, comparison_operator,
                            period=60, evaluation_periods=5):
    try:
        cloudwatch_client.put_metric_alarm(
            AlarmName=alarm_name,
            AlarmDescription=f"Alarm for {metric_name} on SageMaker endpoint {ENDPOINT_NAME}",
            ActionsEnabled=False,
            MetricName=metric_name,
            Namespace=namespace,
            Statistic=statistic,
            Dimensions=dimensions,
            Period=period,
            EvaluationPeriods=evaluation_periods,
            Threshold=threshold,
            ComparisonOperator=comparison_operator,
            TreatMissingData='missing'
        )
        print(f"Successfully created alarm: {alarm_name}")
    except Exception as e:
        print(f"Error creating alarm {alarm_name}: {e}")

In [25]:
# Create cloudwatch alarms for high CPU usage, high latency, and endpoint errors
create_cloudwatch_alarm(
    alarm_name=f"DDoS-Endpoint-High-CPU-{ENDPOINT_NAME}",
    metric_name="CPUUtilization",
    namespace="AWS/SageMaker",
    dimensions=endpoint_dimensions,
    statistic="Average",
    threshold=80.0,
    comparison_operator="GreaterThanOrEqualToThreshold"
)

create_cloudwatch_alarm(
    alarm_name=f"DDoS-Endpoint-High-Latency-{ENDPOINT_NAME}",
    metric_name="ModelLatency",
    namespace="AWS/SageMaker",
    dimensions=endpoint_dimensions,
    statistic="Average",
    threshold=5000,
    comparison_operator="GreaterThanOrEqualToThreshold"
)

create_cloudwatch_alarm(
    alarm_name=f"DDoS-Endpoint-Invocation-Errors-{ENDPOINT_NAME}",
    metric_name="Invocation5XXErrors",
    namespace="AWS/SageMaker",
    dimensions=endpoint_dimensions,
    statistic="Sum",
    threshold=1,
    comparison_operator="GreaterThanOrEqualToThreshold",
    evaluation_periods=1
)

Successfully created/updated alarm: DDoS-Endpoint-High-CPU-final-model-endpoint
Successfully created/updated alarm: DDoS-Endpoint-High-Latency-final-model-endpoint
Successfully created/updated alarm: DDoS-Endpoint-Invocation-Errors-final-model-endpoint


In [26]:
# Configure Cloudwatch dashboard for CPU and memory utilization, endpoint invocations and errors, and latency
dashboard_body = {
    "widgets": [
        {
            "type": "metric",
            "x": 0, "y": 0, "width": 12, "height": 6,
            "properties": {
                "metrics": [
                    ["AWS/SageMaker", "CPUUtilization", "EndpointName", ENDPOINT_NAME, "VariantName", "AllTraffic",
                     {"stat": "Average"}]
                ],
                "view": "timeSeries", "stacked": False, "region": region,
                "title": "CPU Utilization (%)"
            }
        },
        {
            "type": "metric",
            "x": 12, "y": 0, "width": 12, "height": 6,
            "properties": {
                "metrics": [
                    ["AWS/SageMaker", "MemoryUtilization", "EndpointName", ENDPOINT_NAME, "VariantName", "AllTraffic",
                     {"stat": "Average"}]
                ],
                "view": "timeSeries", "stacked": False, "region": region,
                "title": "Memory Utilization (%)"
            }
        },
        {
            "type": "metric",
            "x": 0, "y": 6, "width": 12, "height": 6,
            "properties": {
                "metrics": [
                    ["AWS/SageMaker", "Invocations", "EndpointName", ENDPOINT_NAME, "VariantName", "AllTraffic",
                     {"stat": "Sum"}]
                ],
                "view": "timeSeries", "stacked": False, "region": region,
                "title": "Total Invocations (Count)"
            }
        },
        {
            "type": "metric",
            "x": 12, "y": 6, "width": 12, "height": 6,
            "properties": {
                "metrics": [
                    ["AWS/SageMaker", "ModelLatency", "EndpointName", ENDPOINT_NAME, "VariantName", "AllTraffic",
                     {"stat": "Average"}],
                    [".", "OverheadLatency", ".", ".", ".", ".", {"stat": "Average"}]
                ],
                "view": "timeSeries", "stacked": False, "region": region,
                "title": "Model and Overhead Latency (ms)"
            }
        },
        {
            "type": "metric",
            "x": 0, "y": 12, "width": 24, "height": 6,
            "properties": {
                "metrics": [
                    ["AWS/SageMaker", "Invocation4XXErrors", "EndpointName", ENDPOINT_NAME, "VariantName", "AllTraffic",
                     {"stat": "Sum"}],
                    [".", "Invocation5XXErrors", ".", ".", ".", ".", {"stat": "Sum"}]
                ],
                "view": "timeSeries", "stacked": False, "region": region,
                "title": "Invocation Errors (4xx & 5xx)"
            }
        }
    ]
}

In [29]:
# Provision the cloudwatch dashboard
cloudwatch_client.put_dashboard(
    DashboardName=dashboard_name,
    DashboardBody=json.dumps(dashboard_body)
)
dashboard_url = f"https://{region}.console.aws.amazon.com/cloudwatch/home?region={region}#dashboards:name={dashboard_name}"
print(f"Dashboard URL: {dashboard_url}")

Dashboard URL: https://us-east-1.console.aws.amazon.com/cloudwatch/home?region=us-east-1#dashboards:name=SageMaker-Endpoint-Health-final-model-endpoint
